# Phân tích Embedding Space: Teacher vs Student

Notebook chẩn đoán độc lập — chạy sau khi đã có `best_model.pth` của student.

| Cell | Chỉ số | Câu hỏi trả lời |
|---|---|---|
| 3 | Thu thập embeddings | — |
| 4 | **CKA** | 2 không gian có cấu trúc tương đồng không? |
| 5 | **Phân phối cosine** | Student phân biệt same/diff identity tốt như teacher không? |
| 6 | **t-SNE** | Các cụm identity có tách biệt rõ không? |
| 7 | **Intra/Inter class ratio** | Embeddings có compact trong cùng identity không? |
| 8 | **Per-identity alignment** | Identity nào student chưa học được từ teacher? |
| 9 | **Norm distribution** | Student có dùng hết không gian embedding không? |

## 1. Setup môi trường

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_URL    = 'https://github.com/NguyenXuanBinh22/DATN.git'
REPO_BRANCH = 'convnext-v2-dev'
REPO_DIR    = '/content/FR_Photometric_Stereo'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull origin {REPO_BRANCH}')

%cd {REPO_DIR}

os.system('pip install -q albumentations==1.3.1 timm tabulate umap-learn')
print('Setup xong.')

## 2. Imports & Cấu hình

Sửa `STUDENT_CKPT` trỏ đến checkpoint student muốn phân tích.

In [ ]:
%cd /content/FR_Photometric_Stereo
import warnings; warnings.filterwarnings('ignore')

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import albumentations as A
from sklearn.manifold import TSNE
from tabulate import tabulate

from going_modular.dataloader.multitask import create_multitask_datafetcher, create_eval_loaders
from going_modular.model.MTLFaceRecognition import MTLFaceRecognition
from going_modular.model.FaceRecognitionMobileNetV3 import FaceRecognitionMobileNetV3
from going_modular.utils.transforms import RandomResizedCropRect, GaussianNoise

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(f'Device: {device}')

# ────────────────────────────────────────────────
# Sửa các đường dẫn bên dưới
# ────────────────────────────────────────────────
DRIVE_DATASET_DIR = '/content/drive/MyDrive/Photometric_DB_Full/'
TEACHER_CKPT      = '/content/drive/MyDrive/Photometric_DB_Full/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth'
STUDENT_CKPT      = '/content/drive/MyDrive/experiments/KD_ConvNextV2_to_MobileNetV3_Albedo/checkpoints/best_model.pth'

CFG = {
    'dataset_dir':      DRIVE_DATASET_DIR,
    'type':             'albedo',
    'teacher_backbone': 'convnextv2_tiny',
    'backbone':         'mobilenetv3_large_100',
    'use_sampler':      False,
    'device':           device,
    'batch_size':       64,
    'image_size':       112,
    'num_classes':      None,
    'output_dir':       '/content/drive/MyDrive/',
    'note':             'analysis',
    'epochs':           1,
    'base_lr':          1e-4,
}

# Số identity hiển thị trên t-SNE (chọn ngẫu nhiên để plot không bị rối)
N_TSNE_CLASSES = 30

In [ ]:
dataset_dir = CFG['dataset_dir']

train_csv = os.path.join(dataset_dir, 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(dataset_dir, 'train_set.csv')

df_train = pd.read_csv(train_csv)
CFG['num_classes'] = int(df_train['id'].nunique())
print(f'num_classes: {CFG["num_classes"]} | samples: {len(df_train)}')

test_transform = A.Compose([
    A.Resize(CFG['image_size'], CFG['image_size']),
])
train_transform = A.Compose([
    RandomResizedCropRect(CFG['image_size']),
    GaussianNoise(p=0.2),
    A.HorizontalFlip(p=0.5),
])

_, probe_dl_train, _ = create_multitask_datafetcher(
    CFG, train_transform, test_transform, 'train_split.csv', 'probe_split.csv'
)
gallery_dl, probe_dl = create_eval_loaders(CFG, test_transform)
print(f'Gallery batches: {len(gallery_dl)} | Probe batches: {len(probe_dl)}')

In [ ]:
# ── Teacher ──────────────────────────────────────────────────────────────
teacher = MTLFaceRecognition(
    backbone=CFG['teacher_backbone'],
    num_classes=CFG['num_classes'],
)
ckpt = torch.load(TEACHER_CKPT, map_location=device, weights_only=False)
state_dict = ckpt['model_state_dict']
for k in [k for k in state_dict if 'id_head.maglinear' in k]:
    del state_dict[k]
teacher.load_state_dict(state_dict, strict=False)
teacher.to(device).eval()
for p in teacher.parameters(): p.requires_grad = False

class TeacherWrapper(torch.nn.Module):
    def __init__(self, m): super().__init__(); self._m = m
    def get_embedding(self, x): return self._m.get_embedding(x)[-1]

teacher_w = TeacherWrapper(teacher).to(device)
print(f'Teacher loaded — epoch {ckpt.get("epoch", "?")}')

# ── Student ──────────────────────────────────────────────────────────────
student = FaceRecognitionMobileNetV3(
    num_classes=CFG['num_classes'],
    backbone=CFG['backbone'],
)
s_ckpt = torch.load(STUDENT_CKPT, map_location=device, weights_only=False)
student.load_state_dict(s_ckpt['model_state_dict'])
student.to(device).eval()
print(f'Student loaded — epoch {s_ckpt.get("epoch", "?")}')

## 3. Thu thập Embeddings

In [ ]:
@torch.no_grad()
def collect_both(dataloader, teacher_w, student, device):
    """Thu thập (teacher_emb, student_emb, labels) — L2-normalized."""
    t_list, s_list, lbl_list = [], [], []
    for X, y in dataloader:
        X = X.to(device)
        t_list.append(teacher_w.get_embedding(X).cpu())
        s_list.append(student.get_embedding(X).cpu())
        lbl_list.append(y[:, 0])
    T = F.normalize(torch.cat(t_list), p=2, dim=1)
    S = F.normalize(torch.cat(s_list), p=2, dim=1)
    L = torch.cat(lbl_list)
    return T, S, L


print('Thu thập embeddings từ gallery + probe...')
T_g, S_g, L_g = collect_both(gallery_dl, teacher_w, student, device)
T_p, S_p, L_p = collect_both(probe_dl,   teacher_w, student, device)

# Gộp gallery + probe thành tập đầy đủ
T_all = torch.cat([T_g, T_p])
S_all = torch.cat([S_g, S_p])
L_all = torch.cat([L_g, L_p])

print(f'Tổng số mẫu : {len(L_all)}')
print(f'Số identities: {L_all.unique().numel()}')
print(f'Teacher emb  : {T_all.shape}  |  Student emb: {S_all.shape}')

## 4. CKA — Mức độ tương đồng tổng thể

**Centered Kernel Alignment (CKA)** đo lường mức độ cấu trúc (hình học) của 2 không gian embedding giống nhau.  
- `CKA = 1.0` → 2 không gian hoàn toàn tương đồng (student học được đúng cấu trúc của teacher)  
- `CKA < 0.5` → cấu trúc rất khác nhau — KD chưa hiệu quả  

CKA không yêu cầu 2 embedding space phải giống nhau tuyệt đối (vì có thể có rotation), chỉ cần **quan hệ giữa các điểm** tương đồng.

In [ ]:
def linear_cka(X: torch.Tensor, Y: torch.Tensor) -> float:
    """
    Linear CKA trên ma trận Gram.
    X, Y: (N, D) — đã L2-normalize hoặc chưa đều được.
    """
    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)
    XXT = X @ X.T
    YYT = Y @ Y.T
    hsic_xy = (XXT * YYT).sum()
    hsic_xx = (XXT * XXT).sum()
    hsic_yy = (YYT * YYT).sum()
    return (hsic_xy / (hsic_xx.sqrt() * hsic_yy.sqrt())).item()


# CKA trên tất cả mẫu (subsample để tránh OOM nếu dataset lớn)
MAX_CKA = 2000
idx = torch.randperm(len(L_all))[:MAX_CKA]
cka_score = linear_cka(T_all[idx], S_all[idx])

print(f'Linear CKA (teacher vs student): {cka_score:.4f}')
if cka_score >= 0.7:
    print('  → Tốt: student học được cấu trúc của teacher')
elif cka_score >= 0.4:
    print('  → Trung bình: cấu trúc có tương đồng nhưng chưa đủ tốt')
else:
    print('  → Kém: 2 không gian gần như độc lập — KD chưa hiệu quả')

## 5. Phân phối Cosine Similarity

So sánh histogram cosine similarity cho **cặp cùng identity (positive)** vs **cặp khác identity (negative)** giữa teacher và student.

- **Khoảng cách giữa 2 phân phối càng lớn** → model phân biệt identity tốt hơn  
- **Overlap nhỏ** → ít trường hợp nhầm lẫn

In [ ]:
def sample_pairs(emb: torch.Tensor, labels: torch.Tensor, n_pos=5000, n_neg=5000):
    """Lấy mẫu ngẫu nhiên n_pos positive pairs và n_neg negative pairs."""
    N = len(labels)
    # Ma trận nhãn same/diff
    same = (labels.unsqueeze(0) == labels.unsqueeze(1))  # (N, N)
    triu = torch.triu(torch.ones(N, N, dtype=torch.bool), diagonal=1)

    pos_idx = same & triu
    neg_idx = (~same) & triu

    # Cosine sim matrix
    cos_mat = emb @ emb.T  # đã L2-norm → cosine sim

    pos_sims = cos_mat[pos_idx]
    neg_sims = cos_mat[neg_idx]

    # Subsample
    pos_sims = pos_sims[torch.randperm(len(pos_sims))[:n_pos]].numpy()
    neg_sims = neg_sims[torch.randperm(len(neg_sims))[:n_neg]].numpy()
    return pos_sims, neg_sims


t_pos, t_neg = sample_pairs(T_all, L_all)
s_pos, s_neg = sample_pairs(S_all, L_all)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
bins = np.linspace(-1, 1, 80)

for ax, pos, neg, title in [
    (axes[0], t_pos, t_neg, f'Teacher ({CFG["teacher_backbone"]})'),
    (axes[1], s_pos, s_neg, f'Student ({CFG["backbone"]})')
]:
    ax.hist(neg, bins=bins, alpha=0.6, color='#e74c3c', label='Khác identity', density=True)
    ax.hist(pos, bins=bins, alpha=0.6, color='#2ecc71', label='Cùng identity',  density=True)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Cosine Similarity')
    ax.set_ylabel('Density')
    ax.legend()
    ax.axvline(np.mean(pos), color='#27ae60', linestyle='--', linewidth=1.2,
               label=f'μ_pos={np.mean(pos):.3f}')
    ax.axvline(np.mean(neg), color='#c0392b', linestyle='--', linewidth=1.2,
               label=f'μ_neg={np.mean(neg):.3f}')
    ax.legend(fontsize=9)

fig.suptitle('Phân phối Cosine Similarity: Positive vs Negative Pairs', fontsize=14)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/cosine_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Thống kê
rows = [
    ['μ positive', f'{np.mean(t_pos):.4f}', f'{np.mean(s_pos):.4f}'],
    ['μ negative', f'{np.mean(t_neg):.4f}', f'{np.mean(s_neg):.4f}'],
    ['Δ (pos - neg)', f'{np.mean(t_pos)-np.mean(t_neg):.4f}', f'{np.mean(s_pos)-np.mean(s_neg):.4f}'],
    ['std positive', f'{np.std(t_pos):.4f}', f'{np.std(s_pos):.4f}'],
    ['std negative', f'{np.std(t_neg):.4f}', f'{np.std(s_neg):.4f}'],
]
print(tabulate(rows, headers=['Metric', 'Teacher', 'Student'], tablefmt='fancy_grid'))

## 6. t-SNE — Trực quan hóa cụm 2D

Giảm chiều 512-D → 2D bằng t-SNE, tô màu theo identity.  
- **Cụm gọn, tách biệt** → model phân biệt identity tốt  
- **Cụm lẫn lộn** → model chưa học được ranh giới identity

In [ ]:
# Chọn N_TSNE_CLASSES identity và tối đa 20 mẫu mỗi identity
MAX_PER_CLASS = 20
rng = np.random.default_rng(42)

unique_ids = L_all.unique().numpy()
selected_ids = rng.choice(unique_ids, size=min(N_TSNE_CLASSES, len(unique_ids)), replace=False)
selected_ids = sorted(selected_ids)

idx_keep = []
for uid in selected_ids:
    idx_uid = (L_all == uid).nonzero(as_tuple=True)[0].numpy()
    chosen  = rng.choice(idx_uid, size=min(MAX_PER_CLASS, len(idx_uid)), replace=False)
    idx_keep.extend(chosen.tolist())

idx_keep = np.array(idx_keep)
T_sub = T_all[idx_keep].numpy()
S_sub = S_all[idx_keep].numpy()
L_sub = L_all[idx_keep].numpy()

# Remap labels sang 0..N_TSNE_CLASSES-1
id2color = {uid: i for i, uid in enumerate(selected_ids)}
colors   = np.array([id2color[l] for l in L_sub])

print(f'Tính t-SNE cho {len(idx_keep)} mẫu × {N_TSNE_CLASSES} identities...')
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
T_2d = tsne.fit_transform(T_sub)
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
S_2d = tsne.fit_transform(S_sub)

cmap = plt.colormaps.get_cmap('tab20').resampled(N_TSNE_CLASSES)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, emb2d, title in [
    (axes[0], T_2d, f'Teacher ({CFG["teacher_backbone"]})'),
    (axes[1], S_2d, f'Student ({CFG["backbone"]})')
]:
    sc = ax.scatter(emb2d[:, 0], emb2d[:, 1],
                    c=colors, cmap=cmap, s=18, alpha=0.75,
                    vmin=0, vmax=N_TSNE_CLASSES - 1)
    ax.set_title(title, fontsize=13)
    ax.axis('off')

fig.suptitle(f't-SNE — {N_TSNE_CLASSES} identities ngẫu nhiên', fontsize=14)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/tsne_embedding.png', dpi=150, bbox_inches='tight')
plt.show()
print('Đã lưu: /content/drive/MyDrive/tsne_embedding.png')

## 7. Intra / Inter Class Distance Ratio

- **Intra-class distance** (d_intra): khoảng cách cosine trung bình giữa các mẫu **cùng** identity  
- **Inter-class distance** (d_inter): khoảng cách cosine trung bình giữa các mẫu **khác** identity  
- **Ratio = d_intra / d_inter**: càng nhỏ càng tốt (cụm gọn, cách xa nhau)

In [ ]:
def intra_inter_stats(emb: torch.Tensor, labels: torch.Tensor, n_sample=3000):
    """Tính intra/inter class distance từ cosine distance (1 - cosine_sim)."""
    N = len(labels)
    idx = torch.randperm(N)[:n_sample]
    emb, labels = emb[idx], labels[idx]

    cos_mat  = emb @ emb.T                               # cosine sim [n, n]
    dist_mat = 1.0 - cos_mat                             # cosine dist
    triu     = torch.triu(torch.ones(len(labels), len(labels), dtype=torch.bool), diagonal=1)
    same     = (labels.unsqueeze(0) == labels.unsqueeze(1))

    intra = dist_mat[same  & triu]
    inter = dist_mat[(~same) & triu]

    return {
        'intra_mean': intra.mean().item(),
        'intra_std':  intra.std().item(),
        'inter_mean': inter.mean().item(),
        'inter_std':  inter.std().item(),
        'ratio':      (intra.mean() / inter.mean()).item(),
    }


t_stats = intra_inter_stats(T_all, L_all)
s_stats = intra_inter_stats(S_all, L_all)

rows = [
    ['d_intra (mean ± std)',
     f"{t_stats['intra_mean']:.4f} ± {t_stats['intra_std']:.4f}",
     f"{s_stats['intra_mean']:.4f} ± {s_stats['intra_std']:.4f}"],
    ['d_inter (mean ± std)',
     f"{t_stats['inter_mean']:.4f} ± {t_stats['inter_std']:.4f}",
     f"{s_stats['inter_mean']:.4f} ± {s_stats['inter_std']:.4f}"],
    ['Ratio d_intra / d_inter  (↓ tốt)',
     f"{t_stats['ratio']:.4f}",
     f"{s_stats['ratio']:.4f}"],
]
print(tabulate(rows, headers=['Metric', 'Teacher', 'Student'], tablefmt='fancy_grid'))

# Barplot so sánh
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(2)
intra_vals = [t_stats['intra_mean'], s_stats['intra_mean']]
inter_vals = [t_stats['inter_mean'], s_stats['inter_mean']]
w = 0.35
ax.bar(x - w/2, intra_vals, w, label='Intra-class', color='#e74c3c', alpha=0.8)
ax.bar(x + w/2, inter_vals, w, label='Inter-class', color='#3498db', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f'Teacher\n({CFG["teacher_backbone"]})', f'Student\n({CFG["backbone"]})', ])
ax.set_ylabel('Cosine Distance')
ax.set_title('Intra vs Inter Class Distance (cosine)')
ax.legend()
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/intra_inter_distance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Per-Identity Alignment

Với mỗi identity, tính **cosine distance trung bình** giữa teacher embedding và student embedding của các mẫu cùng identity đó.

- **Thấp** → student học tốt identity này từ teacher  
- **Cao** → identity này student chưa align được với teacher — cần điều tra thêm

In [ ]:
unique_ids = L_all.unique().numpy()
per_id_dist = []

for uid in unique_ids:
    mask = (L_all == uid)
    t_emb = T_all[mask]
    s_emb = S_all[mask]
    # Cosine distance giữa teacher và student embedding của cùng identity
    cos_sim = F.cosine_similarity(t_emb, s_emb, dim=1).mean().item()
    per_id_dist.append({'id': uid, 'cos_sim': cos_sim, 'cos_dist': 1 - cos_sim, 'n_samples': mask.sum().item()})

df_align = pd.DataFrame(per_id_dist).sort_values('cos_dist', ascending=False)

print(f'Tổng số identities: {len(df_align)}')
print(f'Cosine distance trung bình toàn bộ: {df_align["cos_dist"].mean():.4f}')
print(f'Std: {df_align["cos_dist"].std():.4f}')

# Top 10 identities align kém nhất
print('\n--- 10 identity align KÉM nhất (cos_dist cao) ---')
print(df_align.head(10).to_string(index=False))

# Top 10 identities align tốt nhất
print('\n--- 10 identity align TỐT nhất (cos_dist thấp) ---')
print(df_align.tail(10).to_string(index=False))

# Histogram phân phối alignment
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df_align['cos_dist'], bins=40, color='#9b59b6', alpha=0.8, edgecolor='white')
ax.axvline(df_align['cos_dist'].mean(), color='red', linestyle='--',
           label=f"μ = {df_align['cos_dist'].mean():.3f}")
ax.set_xlabel('Cosine Distance (teacher vs student, per identity)')
ax.set_ylabel('Số identity')
ax.set_title('Phân phối Per-Identity Alignment')
ax.legend()
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/per_identity_alignment.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Phân phối Embedding Norm (trước L2 normalize)

Norm của embedding **trước** khi L2 normalize phản ánh mức độ "tự tin" của model (liên quan đến MagFace).  
- **Norm thấp** → model không chắc chắn về identity  
- **Norm cao, phân phối hẹp** → model học được tốt

So sánh teacher vs student để xem student có "tự tin" tương đương không.

In [ ]:
@torch.no_grad()
def collect_raw_embeddings(dataloader, teacher_w, student, device):
    """Thu thập embedding CHƯA L2-normalize để đo norm."""
    t_list, s_list = [], []
    for X, y in dataloader:
        X = X.to(device)
        # Teacher raw embedding
        t_raw = teacher_w._m.get_embedding(X)[-1]
        # Student raw embedding (trước BN → sau BN trong embedding module)
        feat  = student.backbone(X)
        s_raw = student.embedding(feat)
        t_list.append(t_raw.cpu())
        s_list.append(s_raw.cpu())
    return torch.cat(t_list), torch.cat(s_list)


T_raw, S_raw = collect_raw_embeddings(gallery_dl, teacher_w, student, device)
t_norms = T_raw.norm(p=2, dim=1).numpy()
s_norms = S_raw.norm(p=2, dim=1).numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

for ax, norms, title, color in [
    (axes[0], t_norms, f'Teacher ({CFG["teacher_backbone"]})', '#3498db'),
    (axes[1], s_norms, f'Student ({CFG["backbone"]})',  '#e67e22'),
]:
    ax.hist(norms, bins=50, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(norms.mean(), color='black', linestyle='--',
               label=f'μ={norms.mean():.2f}  σ={norms.std():.2f}')
    ax.set_title(title)
    ax.set_xlabel('L2 Norm')
    ax.set_ylabel('Count')
    ax.legend()

fig.suptitle('Phân phối Embedding Norm (trước L2 normalize)', fontsize=13)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/embedding_norm_dist.png', dpi=150, bbox_inches='tight')
plt.show()

rows = [
    ['Mean norm', f'{t_norms.mean():.4f}', f'{s_norms.mean():.4f}'],
    ['Std norm',  f'{t_norms.std():.4f}',  f'{s_norms.std():.4f}'],
    ['Min norm',  f'{t_norms.min():.4f}',  f'{s_norms.min():.4f}'],
    ['Max norm',  f'{t_norms.max():.4f}',  f'{s_norms.max():.4f}'],
]
print(tabulate(rows, headers=['Metric', 'Teacher', 'Student'], tablefmt='fancy_grid'))

## 10. Tổng hợp chẩn đoán

| Chỉ số | Kết quả tốt | Cần chú ý nếu |
|---|---|---|
| CKA | ≥ 0.7 | < 0.5 → KD không hiệu quả |
| Δ(μ_pos − μ_neg) | Student ≈ Teacher | Δ_student << Δ_teacher → phân biệt kém |
| Intra/Inter ratio | Student ≤ Teacher | ratio_student > ratio_teacher → cụm không compact |
| Per-identity alignment μ | < 0.3 | > 0.6 → student gần như không học từ teacher |
| Norm distribution | Phân phối hẹp, tương đồng teacher | Norm thấp bất thường → model không tự tin |